# Hunt Autopsy: train the football perception models on a free GPU

Runs on **Kaggle** (Settings → Accelerator → GPU T4 x2 or P100; 30 h/week free) or **Google Colab** (Runtime → Change runtime type → T4 GPU).

What it does:
1. Clones the repo and installs dependencies.
2. Fine-tunes three YOLO11 models on Roboflow's CC-BY football datasets: players/goalkeeper/referee/ball, ball-only, and 32 pitch keypoints.
3. Downloads the openly licensed eval clips and re-runs **eval 11** with the new models.
4. Runs TAPNet (BootsTAPIR) on the GPU for keeper-dive timing.
5. Packs `checkpoints/` for download. Put the files into the repo's `checkpoints/` folder locally.

**Secrets:** add `ROBOFLOW_API_KEY` (free account at roboflow.com → Settings → API key) as a Kaggle Secret (Add-ons → Secrets) or a Colab secret (🔑 icon). Turn on internet access in Kaggle.

In [ ]:
REPO = "https://github.com/ayuboyussuf/inside_the_game.git"   # must be public, or use a token
!git clone -q $REPO hunt && cd hunt && git log --oneline -1
%cd hunt
!pip -q install -e ".[dev,video]" ultralytics "lap>=0.5.12" roboflow "git+https://github.com/roboflow/sports.git" "tapnet @ git+https://github.com/google-deepmind/tapnet.git"
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to a GPU")

In [ ]:
import os
try:                                    # Kaggle
    from kaggle_secrets import UserSecretsClient
    os.environ["ROBOFLOW_API_KEY"] = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
except Exception:
    try:                                # Colab
        from google.colab import userdata
        os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        pass
print("ROBOFLOW_API_KEY set:", bool(os.environ.get("ROBOFLOW_API_KEY")))

## 1. Fine-tune (≈ 1–2 h on a T4 for all three)

In [ ]:
!python -m training.train_football --what players ball pitch

## 2. Re-run eval 11 on real footage with the new models

In [ ]:
!python scripts/fetch_football_clips.py
!FOOTBALL_EVAL_V2=1 python -m evals.football_video
!python -m evals.run_all --no-video

## 3. TAPNet keeper-dive timing (GPU)

In [ ]:
!wget -q -O checkpoints/bootstapir_checkpoint_v2.pt https://storage.googleapis.com/dm-tapnet/bootstap/bootstapir_checkpoint_v2.pt
!python -m perception.keeper_reaction --clip data/raw/football/keller_saves_henry_pk.ogv

## 4. Download the trained models

In [ ]:
!cd checkpoints && zip -q ../football_checkpoints.zip football_*.pt football_pitch_keypoints.json && cd .. && ls -la football_checkpoints.zip
try:
    from google.colab import files; files.download("football_checkpoints.zip")
except Exception:
    print("Kaggle: download football_checkpoints.zip from the Output panel")